# 03. 베이스라인 모델 학습 및 비교
**데이터셋**: Cardiovascular Disease Dataset (Kaggle, 70,000명)  
**목적**: Logistic Regression / Random Forest / XGBoost 성능 비교 후 최적 모델 선정  
**목표**: AUC 0.85 이상  
**참고 논문**: XGBoost 기반 K-Means 군집 분석을 활용한 심장질환 예측 (한국정보전자통신기술학회, 2025)

In [ ]:
# ── 셀 1: 라이브러리 import
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── 셀 2: 전처리 파이프라인 실행
# (02_preprocessing.ipynb 코드 그대로 가져오기)
df = pd.read_csv('/Users/admin/cardiovascular_ml/data/cardio_train.csv', sep=';')
df['age'] = (df['age'] / 365).astype(int)
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

df = df[
    (df['ap_hi'] >= 80)  & (df['ap_hi'] <= 250) &
    (df['ap_lo'] >= 50)  & (df['ap_lo'] <= 200) &
    (df['height'] >= 140) & (df['height'] <= 210) &
    (df['weight'] >= 40)  & (df['weight'] <= 180) &
    (df['bmi'] >= 10)    & (df['bmi'] <= 60)
]

X = df.drop(columns=['id', 'cardio'])
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
scale_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"훈련: {X_train.shape} / 테스트: {X_test.shape}")

In [ ]:
# ── 셀 3: Logistic Regression (베이스라인)
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_pred)
print(f"Logistic Regression AUC: {lr_auc:.4f}")

In [ ]:
# ── 셀 4: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)
print(f"Random Forest AUC: {rf_auc:.4f}")

In [ ]:
# ── 셀 5: XGBoost (메인 모델)
xgb = XGBClassifier(random_state=42, eval_metric='auc', verbosity=0)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_pred)
print(f"XGBoost AUC: {xgb_auc:.4f}")

In [ ]:
# ── 셀 6: 모델 비교표
results = {
    'Logistic Regression': lr_auc,
    'Random Forest': rf_auc,
    'XGBoost': xgb_auc
}

print("=" * 40)
print("모델 성능 비교")
print("=" * 40)
for model, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    status = "✅ 목표 달성" if auc >= 0.85 else "❌ 목표 미달"
    print(f"{model:<25} AUC: {auc:.4f}  {status}")
print("=" * 40)
print(f"목표: AUC 0.85 이상")
